# MFPT vs m0: Network and Reset Protocol Comparison (Julia)

This notebook compares MFPT as a function of m0 across network types and reset protocols,
using only the Julia package functions from src/VoterResetting.jl.

In [ ]:
using Graphs
using Random
using Statistics
using Printf
using Plots

project_root = isdir(joinpath(pwd(), "src")) ? pwd() : dirname(dirname(pwd()))
include(joinpath(project_root, "src", "VoterResetting.jl"))
using .VoterResetting

# Use explicit imports to avoid name ambiguity with other loaded modules.
import .VoterResetting: ComplexParams, hub_reset, first_passage_time_complex

In [5]:
# Parameters
N = 250
r = 0.1
m0_values = collect(range(-0.9, 0.9, length=11))
nsamples = 1000
seed = 1234

Random.seed!(seed)

graphs = Dict(
    "ER" => erdos_renyi(N, 0.1),
    "RRG" => random_regular_graph(N, 6),
    "SF" => barabasi_albert(N, 2),
)

protocols = Dict(
    "High-degree reset" => true,
    "Low-degree reset" => false,
)

println("Setup complete: N=$N, r=$r, points=$(length(m0_values)), nsamples=$nsamples")

Setup complete: N=250, r=0.1, points=11, nsamples=1000


In [ ]:
function mfpt_curve_vs_m0_complex(G, m0_values; r=0.1, nsamples=1000, highest=true)
    mfpt = zeros(length(m0_values))
    total = length(m0_values)
    t0 = time()

    for (i, m0) in enumerate(m0_values)
        params = VoterResetting.ComplexParams(r, Float64(m0))
        reset_protocol = VoterResetting.hub_reset(Float64(m0); highest=highest)
        out = VoterResetting.first_passage_time_complex(
            G,
            params;
            consensus_type=:either,
            nsamples=nsamples,
            reset=reset_protocol,
        )
        mfpt[i] = out.mean_fpt

        elapsed = time() - t0
        rate = i / max(elapsed, 1e-9)
        eta = (total - i) / max(rate, 1e-9)
        @printf("[m0 %d/%d] m0=%.3f | MFPT=%.4f | elapsed=%.1fs | eta=%.1fs\n", i, total, m0, mfpt[i], elapsed, eta)
        flush(stdout)
    end

    return mfpt
end

mfpt_curve_vs_m0_complex (generic function with 1 method)

In [7]:
results = Dict{String, Dict{String, Vector{Float64}}}()

for (gname, G) in graphs
    results[gname] = Dict{String, Vector{Float64}}()
    for (pname, highest) in protocols
        println("Running: $gname | $pname")
        results[gname][pname] = mfpt_curve_vs_m0_complex(
            G,
            m0_values;
            r=r,
            nsamples=nsamples,
            highest=highest,
        )
    end
end

println("All scans finished.")

Running: RRG | Low-degree reset


UndefVarError: UndefVarError: `ComplexParams` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.

In [8]:
# Figure 1: protocol effect within each network
p1 = plot(layout=(1, 3), size=(1500, 420), legend=:topright)

for (idx, gname) in enumerate(["ER", "RRG", "SF"])
    plot!(p1[idx], m0_values, results[gname]["High-degree reset"];
        lw=2, marker=:circle, label="High-degree reset")
    plot!(p1[idx], m0_values, results[gname]["Low-degree reset"];
        lw=2, ls=:dash, marker=:square, label="Low-degree reset")
    xlabel!(p1[idx], "m0")
    ylabel!(p1[idx], "MFPT")
    title!(p1[idx], gname)
end

plot!(p1, plot_title="MFPT vs m0 by network (r=$r, N=$N)")
display(p1)

# Figure 2: network effect within each protocol
p2 = plot(layout=(1, 2), size=(1100, 420), legend=:topright)

for (idx, pname) in enumerate(["High-degree reset", "Low-degree reset"])
    for gname in ["ER", "RRG", "SF"]
        plot!(p2[idx], m0_values, results[gname][pname];
            lw=2, marker=:circle, label=gname)
    end
    xlabel!(p2[idx], "m0")
    ylabel!(p2[idx], "MFPT")
    title!(p2[idx], pname)
end

plot!(p2, plot_title="MFPT vs m0 by protocol (r=$r, N=$N)")
display(p2)

KeyError: KeyError: key "ER" not found